# 计算引擎

QuantStudio 提供了多种计算引擎来调度和执行计算图。所有引擎均继承自 `QuantStudio.Core.CalcEngine.Engine`，核心入口方法为 `run`，内部按 **初始化 → 准备 → 计算** 三个阶段执行。

## 引擎类层次

```
Engine (顺序引擎, BFS)
├── StackEngine (栈式引擎, DFS init + DFS forward/backward)
├── ParallelEngine (多进程并行引擎)
└── TreeEngine (树形并发引擎)
```

各引擎的适用场景和特点：

| 引擎 | 并发方式 | 遍历策略 | 适用场景 |
|------|---------|---------|--------|
| Engine | 仅 IO 阶段支持线程并发 | BFS init + 递归 compute | 简单任务、调试、单步执行 |
| StackEngine | 仅 IO 阶段支持线程并发 | DFS init + DFS 显式栈 forward/backward | 需精确控制 forward/backward 流程, 或需追踪遍历路径 |
| ParallelEngine | 多进程，每个进程执行全部节点 | BFS + merge_result 合并 | 节点内有重计算且支持 split/merge 的场景 |
| TreeEngine | 多线程/多进程，节点级别并发 | forward/backward 异步并发 | 计算图较深、节点间可并发的复杂 DAG |

In [1]:
import warnings
warnings.filterwarnings(action="ignore")
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [2]:
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

---

## Engine — 基础顺序引擎

`Engine` 是所有计算引擎的基类，提供了标准的三阶段执行流程。它按广度优先顺序遍历计算图，`compute` 阶段按节点列表顺序依次执行。

### 参数

In [3]:
from QuantStudio.Core.CalcEngine import Engine

display(Markdown(Engine().Args.info()))

* IOConcurrentNum(IO并发数): typing.Optional[int], 默认值 None, 在准备计算阶段, 允许的最大IO并发数, None 不限制上限, 根据待准备节点数量自动决定并发度, 当前取值: None

### 三阶段执行流程

`Engine.run()` 按顺序执行以下三个阶段，每个阶段都有日志记录和计时：

#### 1. init — 初始化阶段 (BFS)

从给定的节点列表出发，**广度优先**遍历 DAG。使用队列 (`pop(0)` + 追加队尾) 实现：
- 将每个节点注册到 `context.NodeDict` 中
- 调用 `init_compute(path, init_data, context)`，该调用通常会：
  - 填充 `context.NodeState`（节点运行状态）
  - 填充 `context.PrepareNodeDict`（需要准备数据的节点列表）
- 若 `init_compute` 返回非空列表，将该节点的依赖 (`Deps`) 加入遍历队列
- 路径通过 `path` 参数累积传递，格式为 `[root_qsid, ..., current_qsid]`

#### 2. prepare — 准备阶段

对 `context.PrepareNodeDict` 中注册的节点执行 `prepare_compute(prepare_data, context)`：
- `IOConcurrentNum <= 1`：顺序执行
- `IOConcurrentNum > 1`：使用 `ThreadPoolExecutor` 并发执行，适用于 IO 密集型操作（如数据库读取、缓存加载）
- 若 `PrepareNodeDict` 为空（所有节点均无需准备），直接跳过

#### 3. compute — 正式计算阶段

顺序调用每个目标节点的 `Node.compute(path, fwd_data, context)`：
- `Node.compute` 内部调用 `forward_compute` 收集局部上下文
- 递归调用依赖节点的 `compute`
- 调用 `backward_compute` 执行实际计算
- 返回结果列表，顺序与 `node_list` 一致

In [4]:
print(qs_help(Engine.run))
print("\n" + "-" * 20 + "\n")
print(qs_help(Engine.init))
print("\n" + "-" * 20 + "\n")
print(qs_help(Engine.compute))
print("\n" + "-" * 20 + "\n")
print(qs_help(Engine.prepare))
print("\n" + "-" * 20 + "\n")
print(qs_help(Engine.__enter__))

类型: function
模块: QuantStudio.Core.CalcEngine
签名: Engine.run(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None, fwd_data_list: Optional[List[Any]] = None) -> List[Any]
说明文档:
    给定节点列表, 执行所有节点的计算, 返回每个节点的计算结果
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文对象
        init_data_list: 初始化数据列表
        fwd_data_list: 前向计算输入数据列表
    
    Returns:
        节点计算的结果列表

--------------------

类型: function
模块: QuantStudio.Core.CalcEngine
签名: Engine.init(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None)
说明文档:
    初始化计算图: BFS 遍历依赖树, 注册节点并递归初始化.
    
    从给定节点列表开始, 广度优先遍历每个节点的依赖 (Deps), 将节点注册到上下文的
    NodeDict 中, 并调用每个节点的 init_compute 方法。若某节点的 init_compute 返回了
    初始化数据列表, 则将其依赖节点加入遍历队列, 实现递归初始化。
    
    Args:
        node_list: 待初始化的节点列表 (计算目标)
        context: 全局上下文, 初始化后的节点将注册到 context.NodeDict 中
        init_data_

### 上下文管理器

`Engine` 支持 `with` 语句，进入时将自己压入全局引擎栈 `__QS_Engine__`，退出时弹出。这使得嵌套计算中可以临时切换引擎。

- `__enter__`：将当前引擎实例追加到 `__QS_Engine__` 列表末尾
- `__exit__`：从 `__QS_Engine__` 列表末尾弹出当前引擎实例，不抑制任何异常

```python
with Engine() as eng:
    eng.run(...)
# 等价于:
eng = Engine()
__QS_Engine__.append(eng)
try:
    eng.run(...)
finally:
    __QS_Engine__.pop()
```

支持嵌套使用多个不同引擎：

```python
with Engine() as eng1:
    with StackEngine() as eng2:
        ...  # 当前活动引擎为 eng2
    ...  # 当前活动引擎恢复为 eng1
```

### 示例：使用 Engine 顺序执行计算

以四则运算计算图为例：

In [5]:
from typing import Any, List, Optional
import numpy as np

from QuantStudio.Core.Node import Node, Context
from QuantStudio.Core.CalcEngine import Engine

class Num(Node):
    """数字"""
    def __init__(self, value: float, deps: List["Node"] = [], args: dict = {}, config_file: Optional[str] = None, **kwargs):
        if "Name" not in args:
            args = args | {"Name": str(value)}
        self._Value = value
        return super().__init__(deps=deps, args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context: Any = None) -> Any:
        return self._Value


class Sum(Node):
    """加法"""
    def __init__(self, deps: List["Node"] = [], args: dict = {}, config_file: Optional[str] = None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "sum"} | args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context: Any = None) -> Any:
        return np.sum(bwd_data_list)


class Prod(Node):
    """乘法"""
    def __init__(self, deps: List["Node"] = [], args: dict = {}, config_file: Optional[str] = None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "prod"} | args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context: Any = None) -> Any:
        return np.prod(bwd_data_list)


# 构建计算图: (1 + 2) * 3
Node1 = Prod([Sum([Num(1), Num(2)]), Num(3)], args={"Name": "(1 + 2) * 3"})

eng = Engine()
Rslt = eng.run([Node1], Context())
print(f"{Node1.Name} = {Rslt[0]}")

(1 + 2) * 3 = 9


---

## StackEngine — 栈式引擎

`StackEngine` 继承自 `Engine`，与基类的核心区别在于 **init 和 compute 阶段均采用深度优先遍历 (DFS)**，而非基类的广度优先 (BFS)。全流程 DFS 保证了节点从初始化到计算的生命周期顺序一致。

### 四阶段执行流程

`StackEngine.run()` 顺序执行以下四个阶段：

#### 1. init (DFS)

**覆写了** `Engine.init`，使用**栈**进行深度优先遍历：
- 从目标节点出发，沿**第一个依赖**深探到底部叶子节点
- 回溯后再处理第二、第三等兄弟分支
- 遍历顺序与 forward_compute 的 DFS 顺序一致

核心实现：`Deps[::-1]` 反转依赖顺序后压栈，第一个依赖最后入栈 → 最先出栈 → 优先深探。

#### 2. prepare

复用 `Engine.prepare`，对 `context.PrepareNodeDict` 中注册的所有节点执行 IO 准备，支持线程池并发。

#### 3. 前向计算 (forward, DFS)

从目标节点沿依赖链向下深入：
- 调用 `forward_compute(path, fwd_data, context)` 收集每层局部上下文
- 节点按访问顺序压入 `NodeStack`
- 若 `forward_compute` 返回空列表 → 终止该分支的前向传播

#### 4. 后向计算 (backward)

从 `NodeStack` 末尾（最深的叶子节点）逆序弹出：
- 调用 `backward_compute(path, bwd_data_list, context, local_context)` 执行计算
- 子节点结果通过 `DataStack` 栈式传递给父节点
- 最终每个目标节点在 `DataStack` 中留下一个结果

### 遍历顺序示意

以 `Root → [Left, Right]`，`Left → [Leaf]` 为例：

```
init (DFS):     Root → Left → Leaf → Right
                 └─ 沿 Left 分支深探到底 ─┘   └─ 回溯

forward (DFS):  Root → Left → Leaf → Right   (同序压入 NodeStack)

backward:       Leaf → Left → Right → Root   (逆序出栈, 子→父)
```

### 与 Engine 的差异

| 方面 | Engine | StackEngine |
|------|--------|-------------|
| init 遍历 | BFS 广度优先 | DFS 深度优先 |
| 遍历实现 | 队列 `pop(0)` + 追加队尾 | 栈 `pop()` + 反转压栈 |
| compute 方式 | 递归调用 `Node.compute` | 显式栈 forward/backward |
| 路径追踪 | 递归调用栈隐式 | `NodeStack` / `PathStack` 可检查 |
| 适用场景 | 通用顺序执行 | 需精确控制 forward/backward 数据流 |

In [6]:
from QuantStudio.Core.CalcEngine import StackEngine

print(qs_help(StackEngine.init))
print("\n" + "-" * 20 + "\n")
print(qs_help(StackEngine.run))

类型: function
模块: QuantStudio.Core.CalcEngine
签名: StackEngine.init(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None)
说明文档:
    初始化计算图: DFS 遍历依赖树, 注册节点并递归初始化.
    
    使用栈进行深度优先遍历, 遍历顺序与 forward_compute 的 DFS 顺序一致:
    从目标节点出发, 沿第一个依赖深入到底后再处理其他分支.
    
    Args:
        node_list: 待初始化的节点列表 (计算目标)
        context: 全局上下文, 初始化后的节点将注册到 context.NodeDict 中
        init_data_list: 与 node_list 一一对应的初始化数据列表, None 时使用默认值

--------------------

类型: function
模块: QuantStudio.Core.CalcEngine
签名: StackEngine.run(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None, fwd_data_list: Optional[List[Any]] = None) -> List[Any]
说明文档:
    执行栈式计算: init (DFS) → prepare → forward (DFS) → backward.
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文
        init_data_list: 与 node_list 一一对应的初始化数据列表
        fwd_data_list: 与 node_list 一一对应的

### 示例：使用 StackEngine 栈式计算

同样使用四则运算计算图，对比 forward/backward 的栈遍历顺序：



In [7]:
from QuantStudio.Core.CalcEngine import StackEngine

# 复用前面定义的计算图: (1 + 2) * 3
# 依赖结构: Prod → [Sum → [Num(1), Num(2)], Num(3)]

StackEng = StackEngine()
Rslt = StackEng.run([Node1], Context())
print(f"{Node1.Name} = {Rslt[0]}")

# StackEngine 的 DFS init 顺序: Prod → Sum → Num(1) → Num(2) → Num(3)
# forward (DFS): 同序压栈
# backward: Num(1) → Num(2) → Sum → Num(3) → Prod (逆序出栈)

(1 + 2) * 3 = 9


---

## ParallelEngine — 多进程并行引擎

`ParallelEngine` 继承自 `Engine`，重写了 `compute` 方法。它通过**多进程**并行执行计算：

1. 将 `context` 按进程数 `nTask`（由 `context.PIDList` 决定）**拆分**为多个子上下文
2. 将前向数据也按 `nTask` 拆分（如果数据对象支持 `split` 方法）
3. 每个子进程内**完整执行**所有节点的 `compute`，各子进程独立运行
4. 子进程通过 `Sub2MainQueue` 队列与主进程通信：
   - `"Event"` 消息：同步事件（如信号量累加）
   - `"Calc"` 消息：单个节点计算完成，回传结果
   - `"Done"` 消息：子进程结束，回传更新的上下文数据
5. 主进程收集所有子进程的结果，调用各节点的 `merge_result(result_list, context)` 合并

### 适用条件

- 节点的 `compute` 方法必须支持数据的**拆分-合并**模式（`split`/`merge_result`）
- 适用于每个节点内计算量大、可通过数据分片并行的场景（如大规模因子计算）
- 不适用于节点间拓扑并行的场景——每个子进程都执行全部节点

In [8]:
from QuantStudio.Core.ParallelEngine import ParallelEngine

print(qs_help(ParallelEngine.run))
print("\n" + "-" * 20 + "\n")
print(qs_help(ParallelEngine.compute))

类型: function
模块: QuantStudio.Core.ParallelEngine
签名: ParallelEngine.run(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None, fwd_data_list: Optional[List[Any]] = None) -> List[Any]
说明文档:
    给定节点列表, 执行所有节点的计算, 返回每个节点的计算结果
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文对象
        init_data_list: 初始化数据列表
        fwd_data_list: 前向计算输入数据列表
    
    Returns:
        节点计算的结果列表

--------------------

类型: function
模块: QuantStudio.Core.ParallelEngine
签名: ParallelEngine.compute(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, fwd_data_list: Optional[List[Any]] = None)
说明文档:
    执行正式计算: 对每个目标节点调用 compute 方法并收集结果.
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文
        fwd_data_list: 与 node_list 一一对应的前向计算输入数据, None 时使用默认值
    
    Returns:
        List[Any]: 每个节点 compute 的返回值列表, 顺序与 node_list 一致


---

## TreeEngine — 树形并发引擎

`TreeEngine` 是功能最丰富的引擎，实现了**节点级别**的异步并发计算。它同样基于 forward/backward 两阶段模式，但通过线程池/进程池实现了计算的并发执行。

### 参数

In [9]:
from QuantStudio.Core.TreeEngine import TreeEngine

display(Markdown(TreeEngine().Args.info()))

* IOConcurrentNum(IO并发数): typing.Optional[int], 默认值 None, 在准备计算阶段, 允许的最大IO并发数, None 不限制上限, 根据待准备节点数量自动决定并发度, 当前取值: None
* CalcConcurrentMode(并发模式): typing.Literal['Thread', 'Process'], 默认值 'Process', 当前取值: 'Process'
* CalcConcurrentNum(计算并发数): typing.Optional[int], 默认值 None, 当前取值: None

### 核心机制

#### 节点状态机

每个节点在计算过程中处于以下四种状态之一（`NodeStatus` 枚举）：

| 状态 | 含义 |
|------|------|
| `UNSTARTED` | 从未运行过 |
| `PENDING` | 暂时挂起（等待依赖完成或并发槽位） |
| `RUNNING` | 正在运行 `backward_compute` |
| `DONE` | 运行结束 |

#### 前向传播（Forward）

从根节点沿依赖边向下传播，调用 `forward_compute`：
- 如果返回非空前向数据列表 → 节点标记为 `PENDING`，其依赖节点加入前向队列
- 如果返回空列表 → 节点已是叶子（或选择终止传播），如果有空闲并发槽位则直接提交后向计算

**去重机制**：相同 QSID 的节点如果在 `RUNNING` 或 `PENDING` 状态，新的前向任务会被挂起到 `QSID2PendingFwdTask`，等待该节点 `DONE` 后再处理。

#### 后向传播（Backward）

从前向传播到达的叶子节点开始，调用 `backward_compute` 并向上汇聚：
1. 叶子节点的后向计算在线程/进程池中异步执行
2. 子节点完成后，将结果存入父节点的 `Path2BwdDataList`
3. 当父节点的**所有依赖子节点**都完成后，触发父节点的后向计算
4. 逐层向上汇聚，直至根节点

#### 并发控制

- 使用 `CalcConcurrentNum` 限制同时运行的后向计算任务数
- 超过并发限制时，任务进入 `PendingBwdTask` 队列
- 每当有任务完成释放槽位，立即从等待队列中取出新任务执行

### 两种并发模式

`CalcConcurrentMode` 支持两种模式：

#### Thread 模式（`CalcConcurrentMode="Thread"`）

- 使用 `ThreadPoolExecutor` 执行后向计算
- 线程间共享同一个 `context` 对象，无需序列化
- 适用于 IO 密集型或 Python GIL 不构成瓶颈的计算

#### Process 模式（`CalcConcurrentMode="Process"`，默认）

- 使用自定义 `ProcessPoolExecutor` 执行后向计算
- 每个子进程维护自己的 `__PROC_CONTEXT__` 副本
- 子进程完成后通过 `context.getUpdateData()` / `context.updateContext()` 同步状态变更
- 适用于 CPU 密集型计算，可绕过 GIL

### 执行流程示意

```
DAG:        Root
           /    \
         A       B
        / \       \
       C   D       E

时间线:
t1: forward → 发现 C, D, E 是叶子, 提交 backward(C), backward(D), backward(E)
t2: C 完成 → 检查父节点 A, A 的依赖 [C,D] 未全完成, 等待
t3: D 完成 → A 的全部依赖完成! 提交 backward(A)
    E 完成 → B 的全部依赖完成! 提交 backward(B)
t4: A 完成 → 检查父节点 Root, Root 的依赖 [A,B] 未全完成, 等待
t5: B 完成 → Root 的全部依赖完成! 提交 backward(Root)
t6: Root 完成 → 返回结果

并发度: 3 (C, D, E 同时计算)
```

### 示例：使用 TreeEngine 并发计算


In [10]:
from QuantStudio.Core.TreeEngine import TreeEngine

# 构建较深的计算图
NodeA = Prod([Sum([Num(1), Num(2)]), Num(3)], args={"Name": "A: (1+2)*3"})
NodeB = Prod([Sum([Num(4), Num(5)]), Num(2)], args={"Name": "B: (4+5)*2"})
NodeRoot = Sum([NodeA, NodeB], args={"Name": "Root: A + B"})

# 使用 TreeEngine 执行（Thread 模式）
TreeEng = TreeEngine(args={"CalcConcurrentMode": "Thread", "CalcConcurrentNum": 2})
Rslt = TreeEng.run([NodeRoot], Context())
print(f"{NodeRoot.Name} = {Rslt[0]}")

  0% (0 of 11) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--
100% (11 of 11) |########################| Elapsed Time: 0:00:00 Time:  0:00:00


Root: A + B = 27


## 方法调用对比

| 阶段 | Engine | StackEngine | ParallelEngine | TreeEngine |
|------|--------|-------------|----------------|------------|
| 初始化 | `init_compute` BFS 广度优先 | `init_compute` DFS 深度优先 (覆写) | 继承 Engine | `init_compute` BFS + Path2Node |
| 准备 | `prepare_compute` (可 IO 并发) | 继承 Engine (可 IO 并发) | 继承 Engine | 继承 Engine |
| 计算 | `compute` 递归顺序 | `forward_compute` → `backward_compute` 显式栈 DFS | 多进程 `compute` + `merge_result` | `forward_compute` → `backward_compute` 异步并发 |

## 选择指南

根据任务特征选择合适的引擎：

```
是否需要并发？
├── 否 → 是否需要精确控制 forward/backward 流程, 或需追踪遍历路径？
│        ├── 否 → Engine — 最简顺序执行, BFS 遍历, 适合调试和单步执行
│        └── 是 → StackEngine — DFS 栈式全流程, NodeStack/PathStack 可追踪
└── 是 → 并行粒度是什么？
         ├── 数据级并行（同一节点对不同数据分片并行计算）→ ParallelEngine
         │     条件: 节点的 compute 支持 split/merge_result 模式
         └── 节点级并行（DAG 中不同分支的节点同时计算）→ TreeEngine
               条件: DAG 较深, 分支间无依赖, 可多线程/多进程并发
```

**补充说明**：

- **调试阶段**优先使用 `Engine`，它按 `node_list` 顺序依次执行，行为和结果最直观
- **生产环境**的大规模因子计算通常使用 `ParallelEngine` + 多进程数据分片
- **复杂 DAG**（多分支、深层嵌套）且节点间可独立计算时，`TreeEngine` 能最大化并行度
- **自定义计算流程**（如神经网络式的前向/后向传播）使用 `StackEngine`，可精确控制每步的前向和后向数据流

## 全局引擎栈

模块级变量 `__QS_Engine__` 是一个全局列表，初始包含一个默认 `Engine()` 实例。四个引擎类均实现了上下文管理器协议（`__enter__` / `__exit__`），使用 `with` 语句时：

- **进入** `with Engine(...) as eng:` → `eng` 被追加到 `__QS_Engine__` 末尾
- **退出** `with` 块 → `eng` 从 `__QS_Engine__` 末尾弹出

```python
from QuantStudio.Core.CalcEngine import Engine, StackEngine, __QS_Engine__

print(f"默认引擎数: {len(__QS_Engine__)}")  # 1

with Engine() as eng1:
    print(f"压入 eng1 后: {len(__QS_Engine__)}")  # 2
    with StackEngine() as eng2:
        print(f"压入 eng2 后: {len(__QS_Engine__)}")  # 3
        # eng2 (StackEngine) 是当前活动引擎
    print(f"退出 eng2 后: {len(__QS_Engine__)}")  # 2
    # eng1 (Engine) 恢复为当前活动引擎

print(f"退出 eng1 后: {len(__QS_Engine__)}")  # 1
```

这种设计使得在嵌套计算中可以**临时切换引擎**，而无需显式传递引擎引用。外层代码使用默认 Engine，内层需要特殊执行策略的代码块用 `with StackEngine():` 临时切换，退出后自动恢复。